# 25. Benchmark Hygiene

**Tier:** Evaluation & Production
**Estimated time:** 40 minutes
**Prerequisites:** 24
**Priority:** 🟡 Important — contamination discipline is what makes your eval numbers *mean something*, but it's methodology you apply occasionally, not daily. *If skipped, revisit when:* before publishing any model comparison, building a fine-tuning dataset, or generating synthetic eval data (notebook 36).
**Source material:** @theahmadosman — https://x.com/theahmadosman/status/2064724789952958663

## What You'll Learn
- Why training on your test set is a "cardinal sin" — it destroys the test set's purpose
- The seven contamination pathways: exact, near-duplicate, semantic, translation, synthetic-data, retrieval, and prompt-overfitting
- Classical split discipline: split before preprocessing, freeze the protocol before testing
- A pre-publication checklist you can run before trusting any eval number

## Why This Matters
"If a student gets the final exam while studying, a perfect score no longer measures mastery. It measures access." Every eval number in notebook 24 is only meaningful if the golden dataset is actually unseen by the thing being measured. Contamination is quiet — it doesn't crash anything, it just makes your numbers lie, and it's one of the most common ways otherwise-rigorous teams fool themselves.


## The cardinal sin

If a student gets the final exam while studying, a perfect score no longer measures mastery. It measures access.

The same logic applies to any AI system you evaluate. If your golden dataset (or anything close to it) leaked into training data, fine-tuning data, a retrieval corpus, or even a *prompt* the model has seen before, then a high score doesn't mean the system generalizes — it means the system memorized. And because contamination doesn't produce an error or a crash, it's invisible unless you deliberately check for it.

This notebook builds a hands-on demo: a tiny model that "improves" on a benchmark purely by memorizing, with a score identical to real learning — until you test it on genuinely unseen data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)


## Demo — memorization vs. generalization, visualized

We'll simulate a simple lookup-table "model" trained on a set of (question, answer) pairs. A **contaminated eval** reuses some of those exact training pairs; a **clean eval** uses genuinely new pairs the model never saw. Watch what happens to the contaminated score as we let the model memorize more of the training set, versus what happens to the clean score.

In [ ]:
# Ground truth: a function the model needs to actually learn (not just memorize).
def true_function(x):
    return 2 * x + 1

# Training set the "model" gets to see.
train_x = np.arange(0, 50)
train_y = true_function(train_x)

# A lookup-table model that memorizes an increasing fraction of the training set,
# and falls back to a WRONG guess (a constant) for anything it hasn't memorized.
def lookup_model(x, memorized_fraction, fallback_guess=0):
    memorized = set(train_x[: int(len(train_x) * memorized_fraction)])
    return true_function(x) if x in memorized else fallback_guess

# Contaminated eval: reuses training examples (the "final exam leaked to the student").
contaminated_eval_x = train_x[:20]
# Clean eval: brand-new inputs the model never saw during "training".
clean_eval_x = np.arange(100, 120)

fractions = np.linspace(0.0, 1.0, 11)
contaminated_scores, clean_scores = [], []
for frac in fractions:
    contaminated_acc = np.mean([lookup_model(x, frac) == true_function(x) for x in contaminated_eval_x])
    clean_acc = np.mean([lookup_model(x, frac) == true_function(x) for x in clean_eval_x])
    contaminated_scores.append(contaminated_acc)
    clean_scores.append(clean_acc)

print(f"At 100% memorization: contaminated={contaminated_scores[-1]:.0%}, clean={clean_scores[-1]:.0%}")


In [ ]:
%matplotlib inline
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(fractions * 100, np.array(contaminated_scores) * 100, marker="o", label="contaminated eval (leaked training data)")
ax.plot(fractions * 100, np.array(clean_scores) * 100, marker="o", label="clean eval (genuinely unseen)")
ax.set_xlabel("% of training set memorized")
ax.set_ylabel("eval accuracy (%)")
ax.set_title("A model can appear to improve while only memorizing")
ax.legend()
plt.tight_layout()
plt.show()


*The contaminated score climbs to 100% purely from memorization while the clean score stays at 0% — the model learned nothing about `2x + 1`, it just memorized answers. Any report that only shows the top line is measuring access, not mastery.*

## The contamination taxonomy

Contamination isn't just "the exact test question was in training." It has at least seven distinct pathways, roughly in order of how hard they are to catch:

1. **Exact duplication** — the literal eval example appears verbatim in training data. Easiest to catch (hashing), most commonly checked.
2. **Near-duplicate** — a paraphrase, a reformatted version, or the same example with whitespace/casing changes. Hashing misses this; needs fuzzy matching or embedding similarity.
3. **Semantic overlap** — different wording, same underlying fact or reasoning pattern, learned from a *different* source that happens to cover the same ground.
4. **Translation** — the eval example exists in another language in the training corpus; a multilingual model can "leak" across languages.
5. **Synthetic-data contamination** — you generated synthetic training data *with an LLM that had seen your eval set*, so the contamination is laundered through a generation step (ties directly to notebook 36).
6. **Retrieval-corpus contamination** — for RAG systems, the retrieval corpus itself contains the eval answers, so "the system got it right" says nothing about the underlying model or retrieval quality.
7. **Prompt-overfitting** — not data contamination at all, but iterating your *prompt* against the same eval set so many times that the prompt itself becomes tuned to that specific test set (the eval set functions like a training set for prompt engineering).

In [ ]:
# A tiny fuzzy-duplicate detector — the kind of check that catches pathway #2 (near-duplicate)
# that exact hashing (pathway #1) would miss.
import difflib

train_examples = [
    "What is the capital of France?",
    "Explain how photosynthesis works in plants.",
    "List three prime numbers greater than 10.",
]
eval_examples = [
    "what's the capital city of france",          # near-duplicate of train[0]
    "Describe the process of photosynthesis in plants.",  # near-duplicate of train[1]
    "What is the boiling point of water at sea level?",    # genuinely novel
]

def near_duplicate_ratio(a, b):
    return difflib.SequenceMatcher(None, a.lower(), b.lower()).ratio()

THRESHOLD = 0.7
for ev in eval_examples:
    best = max(train_examples, key=lambda tr: near_duplicate_ratio(ev, tr))
    ratio = near_duplicate_ratio(ev, best)
    flag = "CONTAMINATED" if ratio >= THRESHOLD else "clean"
    print(f"[{flag}] ratio={ratio:.2f}  eval={ev!r}\n         closest train={best!r}")


## Classical split discipline

Three habits prevent contamination from creeping in, even when no single check would catch it:

- **Split before preprocessing.** If you deduplicate, clean, or augment data *before* splitting into train/val/test, information can leak across the split (e.g. a dedup step that merges a near-duplicate pair across the boundary). Split first, then process each split independently.
- **Freeze the protocol before testing.** Decide your eval set, scorer, and success threshold *before* looking at results. If you peek at test performance and then adjust the eval to make a model look better (or worse), the eval set has become an implicit training signal — this is prompt-overfitting (#7) in slow motion.
- **Separate dev evals from audit evals.** Keep a small "dev" eval set you can iterate against freely, and a separate "audit" eval set you touch only once, right before shipping. The dev set will inevitably get some prompt-overfitting; the audit set is your honest read on generalization.

## Pre-publication checklist

Run through this before trusting or publishing any eval number:

In [ ]:
CHECKLIST = [
    "Have I checked for exact duplicates between eval set and training/fine-tuning data?",
    "Have I checked for near-duplicates (fuzzy match / embedding similarity), not just exact hashes?",
    "If this is a RAG system, does the retrieval corpus contain the eval answers verbatim?",
    "If I used an LLM to generate synthetic training data, did that LLM ever see the eval set?",
    "Did I freeze the eval set + scorer BEFORE iterating on the prompt/model, or did I peek first?",
    "Is there a held-out 'audit' eval set I have touched at most once?",
    "Am I reporting the audit-eval score, not just the dev-eval score I iterated against?",
]

def run_checklist(answers: dict):
    """answers: {checklist_item: bool}"""
    missing = [item for item in CHECKLIST if not answers.get(item, False)]
    if not missing:
        print("PASS — no known contamination gaps.")
    else:
        print(f"{len(missing)} unchecked item(s) before you can trust this number:")
        for m in missing:
            print(f"  - {m}")
    return not missing

# Example: a report that only checked exact duplication.
example_answers = {CHECKLIST[0]: True}
run_checklist(example_answers)


## Exercises

**Exercise 1 (Warm-up):** Change `fallback_guess` in the memorization demo to `true_function(x) - 1` (a "smart-looking" wrong guess) instead of `0`. Does the clean-eval curve still expose the memorization, or does the illusion get harder to spot?

**Exercise 2 (Apply):** Extend `near_duplicate_ratio` into a batch contamination scanner: given a list of eval examples and a list of training examples, return every eval example whose best match exceeds `THRESHOLD`, sorted by ratio descending.

**Exercise 3 (Extend):** Notebook 36 covers generating synthetic eval/training data. Using the taxonomy above, write out (in comments) the specific contamination check you'd add to a synthetic-data pipeline to catch pathway #5 (synthetic-data contamination) before it ships.


In [ ]:
# Exercise 1: Warm-up
# Task: Change fallback_guess to a near-miss value and re-plot the clean vs contaminated curves.
# Hint: reuse the fractions/lookup_model loop from the demo cell above.

# YOUR CODE HERE


# Exercise 2: Apply
# Task: Write batch_contamination_scan(eval_examples, train_examples, threshold) -> list[dict].
# Hint: reuse near_duplicate_ratio; each result should include eval text, best match, and ratio.

# YOUR CODE HERE


# Exercise 3: Extend
# Task: Sketch the check that would catch synthetic-data contamination (pathway #5) in a
# pipeline that uses an LLM to generate fine-tuning examples.
# Hint: the risk is the GENERATING model having seen your eval set, not the training data itself.

# YOUR CODE HERE


<details>
<summary>Click to reveal solutions</summary>

```python
# Exercise 1
def lookup_model_near_miss(x, memorized_fraction):
    memorized = set(train_x[: int(len(train_x) * memorized_fraction)])
    return true_function(x) if x in memorized else true_function(x) - 1

near_miss_clean_scores = [
    np.mean([lookup_model_near_miss(x, f) == true_function(x) for x in clean_eval_x])
    for f in fractions
]
# The clean-eval accuracy is still 0% (a near-miss guess is still wrong), but a looser
# scorer (e.g. "within 1 of the true answer") WOULD be fooled — showing why scorer choice
# (notebook 24) and contamination checks are two separate lines of defense.

# Exercise 2
def batch_contamination_scan(eval_examples, train_examples, threshold=0.7):
    results = []
    for ev in eval_examples:
        best = max(train_examples, key=lambda tr: near_duplicate_ratio(ev, tr))
        ratio = near_duplicate_ratio(ev, best)
        if ratio >= threshold:
            results.append({"eval": ev, "closest_train": best, "ratio": ratio})
    return sorted(results, key=lambda r: r["ratio"], reverse=True)

print(batch_contamination_scan(eval_examples, train_examples))

# Exercise 3
# Before shipping a synthetic-data pipeline: run the near-duplicate scanner (Exercise 2) with
# eval_examples = your golden eval set and train_examples = the GENERATED synthetic examples,
# not just the raw source corpus. If the generating LLM paraphrased an eval question into a
# "new" synthetic training example, it will show up as a near-duplicate here even though no
# human ever copy-pasted anything.
```
</details>

## Key Takeaways
- Training on your test set is the "cardinal sin" of evaluation — it doesn't crash anything, it just makes the score stop meaning what you think it means.
- Contamination has (at least) seven pathways — exact, near-duplicate, semantic, translation, synthetic-data, retrieval-corpus, and prompt-overfitting — and only the first is caught by simple hashing.
- Split before preprocessing, freeze the eval protocol before testing, and keep a separate audit eval set you touch at most once.
- A model can show a rising benchmark score from pure memorization — always sanity-check against a genuinely unseen slice before trusting the trend.
- Run the pre-publication checklist before reporting any eval number, especially before comparing two model or prompt versions.

## What's Next
Notebook 26 covers LLM-as-judge in depth — the biases (position, verbosity, self-preference) that can silently corrupt the model-graded scorer introduced in notebook 24.
